In [1]:
from preprocess_Descriptorcalc import embed_only, minimize_only

In [3]:
# In a fresh cell, before importing anything else:
import importlib
import full_bbb_workflow
importlib.reload(full_bbb_workflow)

<module 'full_bbb_workflow' from 'C:\\Users\\vivek\\OneDrive\\Desktop\\Notebook\\full_bbb_workflow.py'>

In [4]:
from full_bbb_workflow import (
    run_full_workflow, fit_final_pipeline, get_imputer,
    get_oof_probabilities_for_final_params, score_compound_for_deployment,
)

In [53]:
import pandas as pd
# --- Your Mordred wrapper: one Mol in, one-row DataFrame out ---
def your_mordred_fn(mol):
    from mordred import Calculator, descriptors
    calc = Calculator(descriptors, ignore_3D=False)
    result = calc(mol)
    return pd.DataFrame([result.asdict()])

# --- Score a new compound ---
row = score_compound_for_deployment(
    "CC(C)NCC(O)COC1=CC=C(CC(N)=O)C=C1", pipeline, results["retained_descriptors"], threshold,
    X_train_imp, y_train, oof_proba_train,
    embed_fn=embed_only, minimize_fn=minimize_only, descriptor_fn=your_mordred_fn,
    descriptor_name_map=results["descriptor_name_map"],
)
print(row)

Dropping 1711 descriptor(s) present here but not used by the trained model (not part of the feature-selected set).
{'SMILES': 'CC(C)NCC(O)COC1=CC=C(CC(N)=O)C=C1', 'Curation_Status': 'skipped_no_curate_fn', '3D_Generation_Status': 'ok', 'Prediction': 'BBB-', 'Confidence': 'Poor/Unreliable', 'BBB_plus_Probability_Percent': 59.99, 'Status': 'Success'}


In [19]:
df = pd.read_csv('Trainingdata.csv')

In [21]:
df.head()

,Unnamed: 0,Smiles,ABC,ABCGG,nAcid,nBase,SpAbs_A,SpMax_A,SpDiam_A,SpAD_A,...,TSRW10,MW,AMW,WPath,WPol,Zagreb1,Zagreb2,mZagreb1,mZagreb2,BBB+/BBB-
0,0,O=C(O)c1cc(N=Nc2ccc(S(=O)(=O)Nc3ccccn3)cc2)ccc1O,NaN,NaN,1,0,35.289886,2.380530,4.761059,35.289886,...,63.201012,398.068491,9.477821,2428,42,144.0,165.0,9.590278,6.097222,BBB-
1,1,COC1(NC(=O)C(C(=O)O)c2ccc(O)cc2)C(=O)N2C(C(=O)...,NaN,NaN,4,0,45.430282,2.648849,5.297577,45.430282,...,87.033695,520.101247,9.287522,4114,62,194.0,237.0,13.756944,7.916667,BBB-
2,2,Oc1c(I)cc(Cl)c2cccnc12,NaN,NaN,0,0,16.678194,2.425683,4.851365,16.678194,...,44.825548,304.910439,16.939469,218,21,68.0,81.0,4.805556,2.861111,BBB-
3,3,CCN=C(NC#N)NCCSCc1ncccc1Br,NaN,NaN,0,3,23.641772,2.237342,4.474683,23.641772,...,50.610337,341.030979,9.743742,898,22,82.0,88.0,6.583333,4.694444,BBB-
4,4,CN1CC[C@]23c4c5ccc(OC6OC(C(=O)O)[C@@H](O)[C@H]...,NaN,NaN,1,1,43.435426,2.709582,5.329713,43.435426,...,85.350582,461.168581,7.686143,2850,74,198.0,254.0,11.229167,6.847222,BBB-


In [23]:
config = {
    "mw_col": "MW", "mw_mode": "range", "mw_low": 150, "mw_high": 800,
    "smiles_col": "Smiles", "y_col": "BBB+/BBB-", "positive_label": None,
    "missingness_report_dir": "",
    "missingness_threshold_pct": 70.0,
    "dist_corr_threshold": 0.25,
    "spearman_threshold": 0.85,
    "winsorize": False,         
    "split_method": "scaffold",
    "test_size": 0.15,
    "imputers": ["KNN"],
    "models": ["LightGBM"],
    "cv": 5, "calibration_method": "isotonic",
    "n_jobs": 5,
     "calibration_plot_path": "calibration_plot.png"
}

In [25]:
# --- Training  ---
# results = run_full_workflow(df, config)
# comparison = results["cv_comparison"]
# threshold = comparison.loc[("KNN", "LightGBM"), "threshold"]
# best_params = comparison.loc[("KNN", "LightGBM"), "best_params"]
# pipeline = fit_final_pipeline(results["X_train"], results["y_train"], "KNN", "LightGBM", best_params)

import joblib

config["n_jobs"] = 8
config["models"] = ["LightGBM"]

with joblib.parallel_backend("threading", n_jobs=config["n_jobs"]):
    results = run_full_workflow(df, config)

comparison = results["cv_comparison"]
threshold = comparison.loc[("KNN", "LightGBM"), "threshold"]
best_params = comparison.loc[("KNN", "LightGBM"), "best_params"]
pipeline = fit_final_pipeline(results["X_train"], results["y_train"], "KNN", "LightGBM", best_params)

=== Step 1: MW filtering ===
MW filter (range): 7688 -> 7192 compounds (496 removed)
Target encoding: 'BBB+' -> 1, 'BBB-' -> 0

=== Step 2: Train/test split ===
train: 6112, test: 1080
train class proportions: {1: 0.633, 0: 0.367}
test class proportions:  {1: 0.703, 0: 0.297}

=== Step 3: Winsorization (bounds fit on TRAIN only) ===
Winsorization: skipped (apply=False)

=== Step 4: Feature selection (fit on TRAIN only, post-winsorization) ===
Distance correlation: 504 / 1827 descriptors retained (threshold > 0.25)
Spearman redundancy pruning: 504 -> 115 descriptors (389 dropped for |rho| > 0.85)

=== Steps 5-8: Imputation x Model comparison via single-level CV (impute+grid-search+calibrate+threshold+evaluate on the same cv folds) ===
  [KNN | LightGBM] MCC=0.750+/-0.029  G_mean=0.881+/-0.014  constraint_ok=True
                  threshold  constraint_satisfied  MCC_mean   MCC_std  \
imputer model                                                           
KNN     LightGBM   0.668375    

In [27]:
# retained = results["retained_descriptors"]
# print(f"{len(retained)} descriptors:")
# for name in retained:
#     print(" -", name)

# # # or save to a file for the manuscript/supplement
# # import pandas as pd
# # pd.Series(retained, name="descriptor").to_csv("retained_descriptors_115.csv", index=False)

In [29]:
# --- Reference data, built ONCE ---
y_train = results["y_train"]
imputer = get_imputer("KNN", 42)
X_train_imp = pd.DataFrame(imputer.fit_transform(results["X_train"]), index=results["X_train"].index, columns=results["X_train"].columns)
oof_proba_train = get_oof_probabilities_for_final_params(X_train_imp, y_train, "LightGBM", best_params)

In [69]:
import joblib

lgbm_bundle = {
    "pipeline": pipeline,
    "retained_descriptors": results["retained_descriptors"],
    "threshold": threshold,
    "descriptor_name_map": results["descriptor_name_map"],
    "X_train_imp": X_train_imp,
    "y_train": y_train,
    "oof_proba_train": oof_proba_train,
}
joblib.dump(lgbm_bundle, "lgbm_bbb_model.joblib")
print("Saved lgbm_bbb_model.joblib")

Saved lgbm_bbb_model.joblib


In [55]:
from predict_bbb import load_bundle, score, score_many

bundle = load_bundle("lgbm_bbb_model.joblib")
result = score("CC(C)NCC(O)COC1=CC=C(CC(N)=O)C=C1", bundle)
print(result)


Dropping 1711 descriptor(s) present here but not used by the trained model (not part of the feature-selected set).
{'SMILES': 'CC(C)NCC(O)COC1=CC=C(CC(N)=O)C=C1', 'Curation_Status': 'skipped_no_curate_fn', '3D_Generation_Status': 'ok', 'Prediction': 'BBB-', 'Confidence': 'Poor/Unreliable', 'BBB_plus_Probability_Percent': 59.99, 'Status': 'Success'}


In [43]:
# import matplotlib.pyplot as plt
# import seaborn as sns
# import pandas as pd
# import os

# # Feature names read off the SHAP summary plot
# shap_top_features = [
#     "TopoPSA", "PPSA5", "BCUTi-1l", "GATS3d", "ETA_epsilon_3", "WNSA4",
#     "MDEC-33", "AATS7Z", "MINsOH", "MINdssC", "MDEC-24", "MATS1c",
#     "Kier2", "VSA_EState3", "ETA_shape_y", "RNCS", "Xch-6d", "Xch-5d",
#     "SlogP_VSA2", "ATSC3c"
# ]

# # Known SHAP-label -> actual dataframe column corrections (hyphen vs underscore)
# name_fixes = {
#     "BCUTi-1l": "BCUTi_1l",
#     "MDEC-33": "MDEC_33",
#     "MDEC-24": "MDEC_24",
#     "Xch-6d": "Xch_6d",
#     "Xch-5d": "Xch_5d",
# }

# resolved_features = [name_fixes.get(f, f) for f in shap_top_features]

# # Final check against actual columns
# missing = [f for f in resolved_features if f not in X_train_imp.columns]
# if missing:
#     print("Still not found in X_train_imp:", missing)

# available_features = [f for f in resolved_features if f in X_train_imp.columns]
# print(f"Resolved {len(available_features)} of {len(shap_top_features)} features")

# df_plot = pd.concat([X_train_imp[available_features], y_train.rename("BBB_class")], axis=1)

# out_dir = "shap_feature_distributions"
# os.makedirs(out_dir, exist_ok=True)

# for col in available_features:
#     plt.figure(figsize=(6, 4))
#     sns.histplot(
#         data=df_plot,
#         x=col,
#         hue="BBB_class",
#         bins=30,
#         kde=False,
#         multiple='stack'
#     )
#     plt.title(f"Distribution of {col} by BBB Class")
#     plt.xlabel(col)
#     plt.ylabel("Number of SMILES")
#     plt.tight_layout()

#     safe_name = col.replace("/", "_")
#     save_path = os.path.join(out_dir, f"{safe_name}_distribution.png")
#     plt.savefig(save_path, dpi=300, bbox_inches="tight")
#     plt.show()
#     plt.close()

# print(f"Saved {len(available_features)} PNGs to: {os.path.abspath(out_dir)}")

In [ ]:
# --- Mordred wrapper: one Mol in, one-row DataFrame out ---
def mordred_fn(mol):
    from mordred import Calculator, descriptors
    calc = Calculator(descriptors, ignore_3D=False)
    result = calc(mol)
    return pd.DataFrame([result.asdict()])

In [ ]:
def curation_fn(smiles):
    canonical = canonicalize_smiles(smiles)
    if canonical is None:
        return None, "invalid_smiles"

    # final_curated_df expects a DataFrame -- wrap the single compound
    single_row_df = pd.DataFrame({"canonicalisedSMILES": [canonical]})
    curated_df = final_curated_df(
        single_row_df, smiles_col="canonicalisedSMILES", exclude_simple_inorganic_carbon=True,
    )

    if len(curated_df) == 0:
        # final_curated_df filtered this compound out -- it survived
        # canonicalization but failed one of the curation checks
        return None, "excluded_by_curation"

    return curated_df["canonicalisedSMILES"].iloc[0], "ok"

In [ ]:
@app.route("/predict", methods=["POST"])
def predict_one():
    smiles = request.json["smiles"]
    result = score_compounds_for_deployment(smiles, pipeline, , retained_descriptors, threshold,
    X_train_imp, y_train, oof_proba_train,
    curate_fn=curation_fn,
    embed_fn=embed_only,
    minimize_fn=minimize_only,
    descriptor_fn=your_mordred_fn,
    descriptor_name_map=descriptor_name_map, verbose=False)
    return jsonify(result.iloc[0].to_dict())

@app.route("/predict_batch", methods=["POST"])
def predict_batch():
    file = request.files["csv"]
    df = pd.read_csv(file)
    result = score_compounds_for_deployment(df, pipeline, retained_descriptors, threshold,
    X_train_imp, y_train, oof_proba_train,
    curate_fn=curation_fn,
    embed_fn=embed_only,
    minimize_fn=minimize_only,
    descriptor_fn=your_mordred_fn,
    descriptor_name_map=descriptor_name_map,smiles_col="SMILES", verbose=False)
    return result.to_csv(index=False)